In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path.cwd().resolve().parents[1]

print("Project root:")
print(ROOT)

# CHANGE THIS ONLY IF YOUR MIMIC-IV LOCATION IS DIFFERENT
MIMIC = ROOT / "data" / "raw" / "mimiciv"

print("\nMIMIC-IV path:")
print(MIMIC)

print("\nExists:", MIMIC.exists())

if MIMIC.exists():
    print("\nFolders:")
    for p in MIMIC.iterdir():
        print(p)

Project root:
/Users/chinmay/Clg Stuff/Research Stuff

MIMIC-IV path:
/Users/chinmay/Clg Stuff/Research Stuff/data/raw/mimiciv

Exists: False


In [5]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path.cwd().resolve().parents[1]

print("Project root:")
print(ROOT)

# CHANGE THIS ONLY IF YOUR MIMIC-IV LOCATION IS DIFFERENT
MIMIC = ROOT / "data" / "raw" / "mimiciv"

print("\nMIMIC-IV path:")
print(MIMIC)

print("\nExists:", MIMIC.exists())

if MIMIC.exists():
    print("\nFolders:")
    for p in MIMIC.iterdir():
        print(p)

Project root:
/Users/chinmay/Clg Stuff/Research Stuff

MIMIC-IV path:
/Users/chinmay/Clg Stuff/Research Stuff/data/raw/mimiciv

Exists: False


In [ ]:
required_hosp = [
    hosp / "patients.csv.gz",
    hosp / "admissions.csv.gz",
    hosp / "diagnoses_icd.csv.gz",
    hosp / "labevents.csv.gz",
]

required_icu = [
    ICU / "ICUstays.csv.gz",
    ICU / "chartevents.csv.gz",
    ICU / "d_items.csv.gz",
]

print("Required files:")
for p in required_hosp + required_ICU:
    print(p.name, "->", p.exists())

NameError: name 'hosp' is not defined

In [ ]:
# Load the smaller/core tables first.
# Do NOT load chartevents or labevents completely yet.

patients = pd.read_csv(
    HOSP / "patients.csv.gz"
)

admissions = pd.read_csv(
    HOSP / "admissions.csv.gz"
)

diagnoses = pd.read_csv(
    HOSP / "diagnoses_icd.csv.gz"
)

icustays = pd.read_csv(
    ICU / "icustays.csv.gz"
)

print("patients:", patients.shape)
print("admissions:", admissions.shape)
print("diagnoses:", diagnoses.shape)
print("icustays:", icustays.shape)

In [ ]:
print("Diagnosis columns:")
print(diagnoses.columns.tolist())

print("\nICD-9 rows:", len(diagnoses))

if "icd_version" in diagnoses.columns:
    print("\nICD versions:")
    print(diagnoses["icd_version"].value_counts(dropna=False))

print("\nSample diagnosis rows:")
display(diagnoses.head())

In [ ]:
# Show diagnosis descriptions available in MIMIC-IV

d_icd = pd.read_csv(
    HOSP / "d_icd_diagnoses.csv.gz"
)

print("Diagnosis dictionary:", d_icd.shape)
print(d_icd.columns.tolist())

display(d_icd.head())

In [ ]:
# Search diagnosis descriptions for encephalopathy-related terminology

text_cols = [
    c for c in ["long_title", "short_title"]
    if c in d_icd.columns
]

mask = False

for c in text_cols:
    mask = mask | d_icd[c].astype(str).str.contains(
        "encephal",
        case=False,
        na=False
    )

enceph_codes = d_icd.loc[
    mask,
    ["icd_code", "icd_version"] + text_cols
].drop_duplicates()

print("Encephalopathy-related ICD codes:")
print("Count:", len(enceph_codes))

display(enceph_codes.head(50))

In [ ]:
sepsis_mask = False

for c in text_cols:
    sepsis_mask = sepsis_mask | d_icd[c].astype(str).str.contains(
        "sepsis|septic",
        case=False,
        na=False,
        regex=True
    )

sepsis_codes = d_icd.loc[
    sepsis_mask,
    ["icd_code", "icd_version"] + text_cols
].drop_duplicates()

print("Sepsis-related ICD codes:")
print("Count:", len(sepsis_codes))

display(sepsis_codes.head(100))

In [ ]:
sepsis_pairs = sepsis_codes[
    ["icd_code", "icd_version"]
].drop_duplicates()

septic_diagnoses = diagnoses.merge(
    sepsis_pairs,
    on=["icd_code", "icd_version"],
    how="inner"
)

print("Sepsis diagnosis rows:", len(septic_diagnoses))

print(
    "Unique septic patients:",
    septic_diagnoses["subject_id"].nunique()
)

print(
    "Unique septic admissions:",
    septic_diagnoses["hadm_id"].nunique()
)

In [ ]:
septic_hadm = septic_diagnoses[
    ["subject_id", "hadm_id"]
].drop_duplicates()

septic_icu = icustays.merge(
    septic_hadm,
    on=["subject_id", "hadm_id"],
    how="inner"
)

print("Septic ICU stays:", len(septic_icu))
print(
    "Septic ICU patients:",
    septic_icu["subject_id"].nunique()
)

display(
    septic_icu[
        [
            "subject_id",
            "hadm_id",
            "stay_id",
            "intime",
            "outtime"
        ]
    ].head()
)

In [ ]:
enceph_pairs = enceph_codes[
    ["icd_code", "icd_version"]
].drop_duplicates()

enceph_diagnoses = diagnoses.merge(
    enceph_pairs,
    on=["icd_code", "icd_version"],
    how="inner"
)

print("Encephalopathy diagnosis rows:", len(enceph_diagnoses))

print(
    "Patients with encephalopathy:",
    enceph_diagnoses["subject_id"].nunique()
)

print(
    "Admissions with encephalopathy:",
    enceph_diagnoses["hadm_id"].nunique()
)

In [ ]:
septic_hadm_set = set(
    septic_diagnoses["hadm_id"]
)

enceph_hadm_set = set(
    enceph_diagnoses["hadm_id"]
)

sae_hadm = septic_hadm_set.intersection(
    enceph_hadm_set
)

print("Sepsis admissions:", len(septic_hadm_set))
print("Encephalopathy admissions:", len(enceph_hadm_set))
print("Sepsis + encephalopathy admissions:", len(sae_hadm))

In [8]:
from pathlib import Path
import pandas as pd
import numpy as np

# ============================================================
# 1. FIND PROJECT ROOT ROBUSTLY
# ============================================================

candidates = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd().parent.parent,
    Path.cwd().parent.parent.parent,
]

ROOT = None

for p in candidates:
    if (p / "data" / "raw" / "mimic-iv").exists():
        ROOT = p
        break

if ROOT is None:
    raise FileNotFoundError(
        "Could not find project root containing data/raw/mimic-iv"
    )

MIMIC = ROOT / "data" / "raw" / "mimic-iv"
HOSP = MIMIC / "hosp"
ICU = MIMIC / "icu"

print("PROJECT ROOT:")
print(ROOT)

print("\nMIMIC-IV:")
print(MIMIC)

print("\nHOSP exists:", HOSP.exists())
print("ICU exists:", ICU.exists())

# ============================================================
# 2. CHECK REQUIRED FILES
# ============================================================

required = [
    HOSP / "patients.csv.gz",
    HOSP / "admissions.csv.gz",
    HOSP / "diagnoses_icd.csv.gz",
    HOSP / "d_icd_diagnoses.csv.gz",
    ICU / "icustays.csv.gz",
]

print("\nREQUIRED FILES:")
for f in required:
    print(f.name, "->", f.exists())

missing = [str(f) for f in required if not f.exists()]

if missing:
    raise FileNotFoundError(
        "\nMissing required files:\n" + "\n".join(missing)
    )

# ============================================================
# 3. LOAD CORE TABLES
# ============================================================

patients = pd.read_csv(
    HOSP / "patients.csv.gz"
)

admissions = pd.read_csv(
    HOSP / "admissions.csv.gz"
)

diagnoses = pd.read_csv(
    HOSP / "diagnoses_icd.csv.gz"
)

d_icd = pd.read_csv(
    HOSP / "d_icd_diagnoses.csv.gz"
)

icustays = pd.read_csv(
    ICU / "icustays.csv.gz"
)

print("\nDATASET SIZES")
print("Patients:", patients.shape)
print("Admissions:", admissions.shape)
print("Diagnoses:", diagnoses.shape)
print("ICD dictionary:", d_icd.shape)
print("ICU stays:", icustays.shape)

# ============================================================
# 4. FIND SEPSIS DIAGNOSIS CODES
# ============================================================

text_cols = [
    c for c in ["long_title", "short_title"]
    if c in d_icd.columns
]

sepsis_mask = False

for c in text_cols:
    sepsis_mask = sepsis_mask | d_icd[c].astype(str).str.contains(
        "sepsis|septic",
        case=False,
        na=False,
        regex=True
    )

sepsis_codes = d_icd.loc[
    sepsis_mask,
    ["icd_code", "icd_version"] + text_cols
].drop_duplicates()

print("\nSEPSIS CODES FOUND:", len(sepsis_codes))

display(sepsis_codes.head(30))

# ============================================================
# 5. FIND ENCEPHALOPATHY CODES
# ============================================================

enceph_mask = False

for c in text_cols:
    enceph_mask = enceph_mask | d_icd[c].astype(str).str.contains(
        "encephal",
        case=False,
        na=False,
        regex=True
    )

enceph_codes = d_icd.loc[
    enceph_mask,
    ["icd_code", "icd_version"] + text_cols
].drop_duplicates()

print("\nENCEPHALOPATHY CODES FOUND:", len(enceph_codes))

display(enceph_codes.head(50))

# ============================================================
# 6. BUILD SEPSIS COHORT
# ============================================================

septic_diagnoses = diagnoses.merge(
    sepsis_codes[["icd_code", "icd_version"]].drop_duplicates(),
    on=["icd_code", "icd_version"],
    how="inner"
)

print("\nSEPSIS COHORT")
print("Diagnosis rows:", len(septic_diagnoses))
print(
    "Unique septic patients:",
    septic_diagnoses["subject_id"].nunique()
)
print(
    "Unique septic admissions:",
    septic_diagnoses["hadm_id"].nunique()
)

# ============================================================
# 7. CONNECT SEPSIS TO ICU STAYS
# ============================================================

septic_hadm = septic_diagnoses[
    ["subject_id", "hadm_id"]
].drop_duplicates()

septic_icu = icustays.merge(
    septic_hadm,
    on=["subject_id", "hadm_id"],
    how="inner"
)

print("\nSEPTIC ICU COHORT")
print(
    "Septic ICU patients:",
    septic_icu["subject_id"].nunique()
)
print(
    "Septic ICU stays:",
    septic_icu["stay_id"].nunique()
)

# ============================================================
# 8. BUILD ENCEPHALOPATHY COHORT
# ============================================================

enceph_diagnoses = diagnoses.merge(
    enceph_codes[["icd_code", "icd_version"]].drop_duplicates(),
    on=["icd_code", "icd_version"],
    how="inner"
)

print("\nENCEPHALOPATHY COHORT")
print(
    "Encephalopathy patients:",
    enceph_diagnoses["subject_id"].nunique()
)
print(
    "Encephalopathy admissions:",
    enceph_diagnoses["hadm_id"].nunique()
)

# ============================================================
# 9. SEPSIS + ENCEPHALOPATHY INTERSECTION
# ============================================================

septic_hadm_set = set(
    septic_diagnoses["hadm_id"]
)

enceph_hadm_set = set(
    enceph_diagnoses["hadm_id"]
)

sae_hadm = septic_hadm_set.intersection(
    enceph_hadm_set
)

print("\n========================================")
print("PRELIMINARY SAE COHORT")
print("========================================")

print(
    "Sepsis admissions:",
    len(septic_hadm_set)
)

print(
    "Encephalopathy admissions:",
    len(enceph_hadm_set)
)

print(
    "Sepsis + encephalopathy admissions:",
    len(sae_hadm)
)

# ============================================================
# 10. PATIENT-LEVEL SAE COHORT
# ============================================================

sae_diagnoses = diagnoses[
    diagnoses["hadm_id"].isin(sae_hadm)
].copy()

print(
    "SAE patients:",
    sae_diagnoses["subject_id"].nunique()
)

# ============================================================
# 11. SAE ICU STAYS
# ============================================================

sae_icu = icustays[
    icustays["hadm_id"].isin(sae_hadm)
].copy()

print(
    "SAE ICU patients:",
    sae_icu["subject_id"].nunique()
)

print(
    "SAE ICU stays:",
    sae_icu["stay_id"].nunique()
)

# ============================================================
# 12. FINAL SUMMARY
# ============================================================

print("\n")
print("=" * 60)
print("FINAL MIMIC-IV COHORT SUMMARY")
print("=" * 60)

print(f"Total MIMIC-IV patients:              {patients.subject_id.nunique():,}")
print(f"Total ICU stays:                       {icustays.stay_id.nunique():,}")
print(f"Septic patients:                       {septic_diagnoses.subject_id.nunique():,}")
print(f"Septic ICU stays:                      {septic_icu.stay_id.nunique():,}")
print(f"Encephalopathy patients:               {enceph_diagnoses.subject_id.nunique():,}")
print(f"Sepsis + encephalopathy admissions:    {len(sae_hadm):,}")
print(f"SAE patients:                          {sae_diagnoses.subject_id.nunique():,}")
print(f"SAE ICU stays:                         {sae_icu.stay_id.nunique():,}")
print("=" * 60)

PROJECT ROOT:
/Users/chinmay/Clg Stuff/Research Stuff/SAE_RL_EarlyWarning

MIMIC-IV:
/Users/chinmay/Clg Stuff/Research Stuff/SAE_RL_EarlyWarning/data/raw/mimic-iv

HOSP exists: True
ICU exists: True

REQUIRED FILES:
patients.csv.gz -> True
admissions.csv.gz -> True
diagnoses_icd.csv.gz -> True
d_icd_diagnoses.csv.gz -> True
icustays.csv.gz -> True

DATASET SIZES
Patients: (100, 6)
Admissions: (275, 16)
Diagnoses: (4506, 5)
ICD dictionary: (109775, 3)
ICU stays: (140, 8)

SEPSIS CODES FOUND: 177


,icd_code,icd_version,long_title
146,67020,9,"Puerperal sepsis, unspecified as to episode of..."
749,73345,9,"Aseptic necrosis of bone, jaw"
1189,0389,9,Unspecified septicemia
1491,0031,9,Salmonella septicemia
1763,0383,9,Septicemia due to anaerobes
1764,03842,9,Septicemia due to escherichia coli [E. coli]
2055,0388,9,Other specified septicemias
2342,03810,9,"Staphylococcal septicemia, unspecified"
2498,73340,9,"Aseptic necrosis of bone, site unspecified"
2569,99802,9,"Postoperative shock, septic"



ENCEPHALOPATHY CODES FOUND: 160


,icd_code,icd_version,long_title
309,05601,9,Encephalomyelitis due to rubella
768,79401,9,Nonspecific abnormal echoencephalogram
890,0722,9,Mumps encephalitis
1191,0463,9,Progressive multifocal leukoencephalopathy
1259,34831,9,Metabolic encephalopathy
1656,76870,9,"Hypoxic-ischemic encephalopathy, unspecified"
2060,0629,9,"Mosquito-borne viral encephalitis, unspecified"
2221,79402,9,Nonspecific abnormal electroencephalogram [EEG]
2944,1390,9,Late effects of viral encephalitis
3105,76871,9,Mild hypoxic-ischemic encephalopathy



SEPSIS COHORT
Diagnosis rows: 52
Unique septic patients: 17
Unique septic admissions: 24

SEPTIC ICU COHORT
Septic ICU patients: 17
Septic ICU stays: 26

ENCEPHALOPATHY COHORT
Encephalopathy patients: 22
Encephalopathy admissions: 23

PRELIMINARY SAE COHORT
Sepsis admissions: 24
Encephalopathy admissions: 23
Sepsis + encephalopathy admissions: 6
SAE patients: 6
SAE ICU patients: 6
SAE ICU stays: 9


FINAL MIMIC-IV COHORT SUMMARY
Total MIMIC-IV patients:              100
Total ICU stays:                       140
Septic patients:                       17
Septic ICU stays:                      26
Encephalopathy patients:               22
Sepsis + encephalopathy admissions:    6
SAE patients:                          6
SAE ICU stays:                         9


In [9]:
from pathlib import Path
import pandas as pd
import numpy as np

# ============================================================
# MIMIC-IV COHORT → SEPSIS + ENCEPHALOPATHY
# ============================================================

ROOT = Path("/Users/chinmay/Clg Stuff/Research Stuff/SAE_RL_EarlyWarning")
MIMIC = ROOT / "data" / "raw" / "mimic-iv"
HOSP = MIMIC / "hosp"
ICU = MIMIC / "icu"

# Load tables
diagnoses = pd.read_csv(HOSP / "diagnoses_icd.csv.gz")
d_icd = pd.read_csv(HOSP / "d_icd_diagnoses.csv.gz")
icustays = pd.read_csv(ICU / "icustays.csv.gz")

# ------------------------------------------------------------
# 1. Find sepsis ICD codes
# ------------------------------------------------------------

text_cols = [c for c in ["long_title", "short_title"] if c in d_icd.columns]

sepsis_mask = False
for c in text_cols:
    sepsis_mask = sepsis_mask | d_icd[c].astype(str).str.contains(
        "sepsis|septic",
        case=False,
        na=False,
        regex=True
    )

sepsis_codes = d_icd.loc[
    sepsis_mask,
    ["icd_code", "icd_version"] + text_cols
].drop_duplicates()

# ------------------------------------------------------------
# 2. Find encephalopathy ICD codes
# ------------------------------------------------------------

enceph_mask = False
for c in text_cols:
    enceph_mask = enceph_mask | d_icd[c].astype(str).str.contains(
        "encephal",
        case=False,
        na=False,
        regex=True
    )

enceph_codes = d_icd.loc[
    enceph_mask,
    ["icd_code", "icd_version"] + text_cols
].drop_duplicates()

# ------------------------------------------------------------
# 3. Build sepsis cohort
# ------------------------------------------------------------

septic_diagnoses = diagnoses.merge(
    sepsis_codes[["icd_code", "icd_version"]].drop_duplicates(),
    on=["icd_code", "icd_version"],
    how="inner"
)

septic_hadm = septic_diagnoses[
    ["subject_id", "hadm_id"]
].drop_duplicates()

septic_icu = icustays.merge(
    septic_hadm,
    on=["subject_id", "hadm_id"],
    how="inner"
)

# ------------------------------------------------------------
# 4. Build encephalopathy cohort
# ------------------------------------------------------------

enceph_diagnoses = diagnoses.merge(
    enceph_codes[["icd_code", "icd_version"]].drop_duplicates(),
    on=["icd_code", "icd_version"],
    how="inner"
)

# ------------------------------------------------------------
# 5. Sepsis + encephalopathy intersection
# ------------------------------------------------------------

sepsis_hadm = set(septic_diagnoses["hadm_id"])
enceph_hadm = set(enceph_diagnoses["hadm_id"])

sae_hadm = sepsis_hadm.intersection(enceph_hadm)

sae_icu = icustays[
    icustays["hadm_id"].isin(sae_hadm)
].copy()

sae_patients = diagnoses[
    diagnoses["hadm_id"].isin(sae_hadm)
]["subject_id"].nunique()

# ------------------------------------------------------------
# 6. Print complete cohort summary
# ------------------------------------------------------------

print("=" * 60)
print("MIMIC-IV SEPSIS-ASSOCIATED ENCEPHALOPATHY COHORT")
print("=" * 60)

print(f"Sepsis ICD codes found:                 {len(sepsis_codes):,}")
print(f"Encephalopathy ICD codes found:         {len(enceph_codes):,}")

print()
print(f"Sepsis diagnosis rows:                  {len(septic_diagnoses):,}")
print(f"Sepsis patients:                        {septic_diagnoses.subject_id.nunique():,}")
print(f"Sepsis admissions:                      {septic_diagnoses.hadm_id.nunique():,}")
print(f"Sepsis ICU stays:                       {septic_icu.stay_id.nunique():,}")

print()
print(f"Encephalopathy diagnosis rows:          {len(enceph_diagnoses):,}")
print(f"Encephalopathy patients:                {enceph_diagnoses.subject_id.nunique():,}")
print(f"Encephalopathy admissions:              {enceph_diagnoses.hadm_id.nunique():,}")

print()
print(f"Sepsis + encephalopathy admissions:     {len(sae_hadm):,}")
print(f"SAE patients:                           {sae_patients:,}")
print(f"SAE ICU stays:                          {sae_icu.stay_id.nunique():,}")

print("=" * 60)

print("\nSample SAE ICU stays:")
display(
    sae_icu[
        ["subject_id", "hadm_id", "stay_id", "intime", "outtime"]
    ].head(20)
)

MIMIC-IV SEPSIS-ASSOCIATED ENCEPHALOPATHY COHORT
Sepsis ICD codes found:                 177
Encephalopathy ICD codes found:         160

Sepsis diagnosis rows:                  52
Sepsis patients:                        17
Sepsis admissions:                      24
Sepsis ICU stays:                       26

Encephalopathy diagnosis rows:          23
Encephalopathy patients:                22
Encephalopathy admissions:              23

Sepsis + encephalopathy admissions:     6
SAE patients:                           6
SAE ICU stays:                          9

Sample SAE ICU stays:


,subject_id,hadm_id,stay_id,intime,outtime
9,10018081,21027282,37293400,2133-12-18 17:10:00,2134-01-01 14:44:53
32,10003400,23559586,38383343,2137-08-17 17:36:37,2137-09-02 19:17:11
33,10004235,24181354,34100191,2196-02-24 17:07:00,2196-02-29 15:58:02
37,10020944,29974575,30757476,2131-02-27 16:40:00,2131-03-08 18:30:38
38,10002428,28662225,38875437,2156-04-19 18:11:19,2156-04-26 18:58:41
53,10002428,28662225,33987268,2156-04-12 16:24:18,2156-04-17 15:57:08
82,10031757,28477280,33244906,2137-10-15 17:29:21,2137-10-17 22:16:51
90,10031757,28477280,30458995,2137-10-12 22:44:57,2137-10-14 17:08:34
131,10003400,23559586,34577403,2137-08-10 19:54:51,2137-08-13 17:54:54
